# Galaxy Pair Inspector
**Validación visual de pares — dataset para entrenamiento ML**

- Descarga imágenes del Legacy Survey DR10 en paralelo (ThreadPoolExecutor).
- Guarda automáticamente las imágenes anotadas de falsos positivos en `outputs/fp_images/`.
- El progreso se persiste en JSON al avanzar cada página.
- Las imágenes guardadas forman el dataset inicial para entrenar un clasificador.

In [1]:
import pandas as pd
import numpy as np
import requests
import json
import os
from io import BytesIO
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

from PIL import Image, ImageDraw

import ipywidgets as widgets
from IPython.display import display, clear_output

from astropy.cosmology import Planck18
import astropy.units as u

print('Entorno listo.')

Entorno listo.


## 1. Configuración

In [2]:
# ── Ruta al catálogo de pares ──────────────────────────────────────────────────
CATALOG_PATH  = '/Users/frank/Documents/Estudio-PhD/Semestre-2025-II/Tesis_I/Galaxy_Pairs/Galaxy_pairs/outputs/catalogs/interacting/DESI_int_legacyID_pairs.parquet'

# ── Salidas ────────────────────────────────────────────────────────────────────
PROGRESS_FILE  = 'outputs/catalogs/progress.json'
OUTPUT_CSV     = 'outputs/catalogs/false_positives.csv'
OUTPUT_CSV_PM  = 'outputs/catalogs/possible_mergers.csv'
OUTPUT_CSV_PAR = 'outputs/catalogs/confirmed_pairs.csv'
FP_IMG_DIR     = 'outputs/fp_images'
PM_IMG_DIR     = 'outputs/pm_images'
PAIR_IMG_DIR   = 'outputs/pair_images'

# ── Filtro de distancia proyectada ────────────────────────────────────────────
RP_MAX_KPC = 12.0

# ── Grid ──────────────────────────────────────────────────────────────────────
GRID_COLS = 4
GRID_ROWS = 2
PAGE_SIZE = GRID_COLS * GRID_ROWS

# ── Legacy Survey ─────────────────────────────────────────────────────────────
LS_LAYER       = 'ls-dr10'
LS_PIXSCALE    = 0.262
IMG_SIZE_PX    = 200
PADDING_FACTOR = 2.5
N_WORKERS      = 16
TIMEOUT        = 8

# ── Estética ──────────────────────────────────────────────────────────────────
CIRCLE_RADIUS = 7
COLOR_G1      = (255, 80,  80)
COLOR_G2      = (80,  180, 255)
TEXT_COLOR    = (255, 255,  50)

## 2. Funciones de imagen

In [3]:
import time

def _adaptive_pixscale(sep_arcsec: float) -> float:
    fov = sep_arcsec * PADDING_FACTOR
    return float(np.clip(fov / IMG_SIZE_PX, 0.3, 2.0))  # mínimo 0.3 evita imágenes negras


def _legacy_url(ra: float, dec: float, pixscale: float) -> str:
    return (f'https://www.legacysurvey.org/viewer/cutout.jpg'
            f'?ra={ra:.6f}&dec={dec:.6f}'
            f'&pixscale={pixscale}&layer={LS_LAYER}&size={IMG_SIZE_PX}')


def _fetch_one(ra1, dec1, ra2, dec2, sep_arcsec) -> Image.Image | None:
    """Descarga un recorte con hasta 3 reintentos. Ejecutada en hilo separado."""
    ra_mid  = (ra1 + ra2) / 2.0
    dec_mid = (dec1 + dec2) / 2.0
    ps      = _adaptive_pixscale(sep_arcsec)
    url     = _legacy_url(ra_mid, dec_mid, ps)
    for intento in range(3):
        try:
            resp = requests.get(url, timeout=TIMEOUT)
            resp.raise_for_status()
            return Image.open(BytesIO(resp.content)).convert('RGB')
        except Exception:
            if intento < 2:
                time.sleep(0.5 * (intento + 1))    # espera 2s antes de reintentar
    return None


def fetch_page_parallel(rows: list[dict]) -> list[Image.Image | None]:
    """
    Descarga todas las imágenes de la página en paralelo.
    Devuelve lista en el mismo orden que rows.
    """
    results = [None] * len(rows)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(_fetch_one,
                        r['ra1'], r['dec1'], r['ra2'], r['dec2'],
                        r['sep_arcsec']): i
            for i, r in enumerate(rows)
        }
        for fut in as_completed(futures):
            results[futures[fut]] = fut.result()
    return results


def _radec_to_pixel(ra, dec, ra_mid, dec_mid, pixscale):
    cos_dec   = np.cos(np.radians(dec_mid))
    dx_arcsec = (ra_mid - ra)  * cos_dec * 3600.0
    dy_arcsec = (dec    - dec_mid) * 3600.0
    cx = IMG_SIZE_PX / 2 + dx_arcsec / pixscale
    cy = IMG_SIZE_PX / 2 - dy_arcsec / pixscale
    return cx, cy


def annotate_image(img: Image.Image, row: pd.Series,
                    rp_col: str | None) -> Image.Image:
    """Dibuja círculos, línea de conexión y etiqueta rp."""
    img  = img.copy()
    draw = ImageDraw.Draw(img)
    ra1, dec1 = row['ra1'], row['dec1']
    ra2, dec2 = row['ra2'], row['dec2']
    sep   = float(row.get('sep_arcsec',
                  np.hypot((ra1-ra2)*np.cos(np.radians((dec1+dec2)/2))*3600,
                           (dec1-dec2)*3600)))
    ra_mid  = (ra1 + ra2) / 2.0
    dec_mid = (dec1 + dec2) / 2.0
    ps      = _adaptive_pixscale(sep)

    x1, y1 = _radec_to_pixel(ra1, dec1, ra_mid, dec_mid, ps)
    x2, y2 = _radec_to_pixel(ra2, dec2, ra_mid, dec_mid, ps)
    r = CIRCLE_RADIUS

    draw.ellipse([x1-r, y1-r, x1+r, y1+r], outline=COLOR_G1, width=2)
    draw.ellipse([x2-r, y2-r, x2+r, y2+r], outline=COLOR_G2, width=2)
    draw.line([x1, y1, x2, y2], fill=(200, 200, 200), width=1)

    label  = f'rp={row[rp_col]:.1f} kpc' if (rp_col and rp_col in row.index) else f'{sep:.1f}"'
    margin = 4
    draw.rectangle([margin-2, IMG_SIZE_PX-18,
                    margin+len(label)*6+2, IMG_SIZE_PX-margin],
                   fill=(0, 0, 0, 160))
    draw.text((margin, IMG_SIZE_PX-18), label, fill=TEXT_COLOR)

    if 'id_par' in row.index:
        draw.text((margin, margin), f'par #{int(row["id_par"])}', fill=TEXT_COLOR)

    return img


def make_error_tile(message: str = 'Sin imagen') -> Image.Image:
    img  = Image.new('RGB', (IMG_SIZE_PX, IMG_SIZE_PX), color=(40, 40, 40))
    draw = ImageDraw.Draw(img)
    draw.text((10, IMG_SIZE_PX//2 - 8), message, fill=(180, 180, 180))
    return img


print('Funciones de imagen definidas.')

Funciones de imagen definidas.


## 3. PairValidator — estado persistente

In [4]:
class PairValidator:
    """
    Gestiona el índice de revisión y tres listas de clasificación:
      - false_positives  → fp_images/
      - possible_mergers → pm_images/
      - confirmed_pairs  → pair_images/
    Los pares cuya imagen no pudo descargarse quedan en pending_retry.
    """

    def __init__(self, catalog_path: str, progress_file: str,
                 fp_img_dir: str, pm_img_dir: str, pair_img_dir: str,
                 rp_max_kpc: float | None = None):
        self.progress_file = progress_file
        self.fp_img_dir    = Path(fp_img_dir)
        self.pm_img_dir    = Path(pm_img_dir)
        self.pair_img_dir  = Path(pair_img_dir)
        for d in (self.fp_img_dir, self.pm_img_dir, self.pair_img_dir):
            d.mkdir(parents=True, exist_ok=True)

        df_full = pd.read_parquet(catalog_path)
        required = {'ra1', 'dec1', 'ra2', 'dec2', 'id1', 'id2'}
        if missing := required - set(df_full.columns):
            raise ValueError(f'Faltan columnas: {missing}')

        for col in ('rp_kpc', 'rp_phys_kpc', 'rp'):
            if col in df_full.columns:
                self.rp_col = col
                break
        else:
            self.rp_col = None

        if rp_max_kpc is not None and self.rp_col:
            self.df = df_full[df_full[self.rp_col] < rp_max_kpc].reset_index(drop=True)
            print(f'Filtro rp < {rp_max_kpc} kpc: {len(df_full):,} → {len(self.df):,} pares')
        else:
            self.df = df_full.reset_index(drop=True)

        self._load_progress()
        print(f'Revisados: {self.current_index:,}  |  '
              f'FP: {len(self.false_positives)}  |  '
              f'Mergers: {len(self.possible_mergers)}  |  '
              f'Pares confirmados: {len(self.confirmed_pairs)}  |  '
              f'Pendientes retry: {len(self.pending_retry)}')

    # ── Persistencia ──────────────────────────────────────────────────────────

    def _load_progress(self):
        if os.path.exists(self.progress_file):
            with open(self.progress_file) as f:
                state = json.load(f)
            self.current_index    = state.get('current_index', 0)
            self.false_positives  = state.get('false_positives', [])
            self.possible_mergers = state.get('possible_mergers', [])
            self.confirmed_pairs  = state.get('confirmed_pairs', [])
            self.pending_retry    = state.get('pending_retry', [])
            print(f'Progreso restaurado desde {self.progress_file}')
        else:
            self.current_index    = 0
            self.false_positives  = []
            self.possible_mergers = []
            self.confirmed_pairs  = []
            self.pending_retry    = []

    def save_progress(self):
        Path(self.progress_file).parent.mkdir(parents=True, exist_ok=True)
        with open(self.progress_file, 'w') as f:
            json.dump({
                'current_index'   : self.current_index,
                'false_positives' : self.false_positives,
                'possible_mergers': self.possible_mergers,
                'confirmed_pairs' : self.confirmed_pairs,
                'pending_retry'   : self.pending_retry,
                'last_saved'      : datetime.now().isoformat(),
            }, f, indent=2)

    def export_csv(self, output_csv: str, output_csv_pm: str, output_csv_par: str):
        for data, path, label in [
            (self.false_positives,  output_csv,     'Falsos positivos'),
            (self.possible_mergers, output_csv_pm,  'Posibles mergers'),
            (self.confirmed_pairs,  output_csv_par, 'Pares confirmados'),
        ]:
            if data:
                pd.DataFrame(data).to_csv(path, index=False)
                print(f'{label} → {path}  ({len(data)} filas)')
            else:
                print(f'No hay {label.lower()} marcados.')

    # ── Navegación ────────────────────────────────────────────────────────────

    def get_page(self, page_size: int) -> pd.DataFrame:
        return self.df.iloc[self.current_index :
                            min(self.current_index + page_size, len(self.df))]

    def advance(self, n: int):
        self.current_index = min(self.current_index + n, len(self.df))

    def go_back(self, n: int):
        self.current_index = max(self.current_index - n, 0)

    # ── Pending retry ─────────────────────────────────────────────────────────

    def add_pending(self, row: pd.Series):
        """Registra un par cuya imagen no pudo descargarse."""
        id1, id2 = int(row['id1']), int(row['id2'])
        already = any(e['id1'] == id1 and e['id2'] == id2 for e in self.pending_retry)
        # tampoco agregar si ya fue clasificado
        classified = (self.is_false_positive(row) or
                      self.is_possible_merger(row) or
                      self.is_confirmed_pair(row))
        if not already and not classified:
            self.pending_retry.append({
                'id1'   : id1,   'id2'  : id2,
                'id_par': int(row['id_par']) if 'id_par' in row.index else None,
                'ra1'   : float(row['ra1']), 'dec1': float(row['dec1']),
                'ra2'   : float(row['ra2']), 'dec2': float(row['dec2']),
                'rp_kpc': float(row[self.rp_col]) if self.rp_col else None,
            })

    def remove_pending(self, row: pd.Series):
        """Elimina de la lista de pendientes (imagen descargada o clasificado)."""
        id1, id2 = int(row['id1']), int(row['id2'])
        self.pending_retry = [
            e for e in self.pending_retry
            if not (e['id1'] == id1 and e['id2'] == id2)
        ]

    def get_pending_df(self) -> pd.DataFrame:
        """Devuelve los pendientes como DataFrame para renderizar."""
        if not self.pending_retry:
            return pd.DataFrame()
        return pd.DataFrame(self.pending_retry)

    # ── Rutas de imagen ───────────────────────────────────────────────────────

    def _par_id(self, row: pd.Series):
        return int(row['id_par']) if 'id_par' in row.index else f"{int(row['id1'])}_{int(row['id2'])}"

    def _fp_img_path(self, row: pd.Series)   -> Path:
        return self.fp_img_dir   / f'par_{self._par_id(row)}.jpg'

    def _pm_img_path(self, row: pd.Series)   -> Path:
        return self.pm_img_dir   / f'par_{self._par_id(row)}.jpg'

    def _pair_img_path(self, row: pd.Series) -> Path:
        return self.pair_img_dir / f'par_{self._par_id(row)}.jpg'

    def _row_record(self, row: pd.Series, img_path: Path) -> dict:
        return {
            'id1'     : int(row['id1']),
            'id2'     : int(row['id2']),
            'id_par'  : int(row['id_par']) if 'id_par' in row.index else None,
            'ra1'     : float(row['ra1']),  'dec1': float(row['dec1']),
            'ra2'     : float(row['ra2']),  'dec2': float(row['dec2']),
            'rp_kpc'  : float(row[self.rp_col]) if self.rp_col else None,
            'img_path': str(img_path),
        }

    # ── Falsos positivos ──────────────────────────────────────────────────────

    def mark_false_positive(self, row: pd.Series, img: Image.Image):
        id1, id2 = int(row['id1']), int(row['id2'])
        if not any(e['id1'] == id1 and e['id2'] == id2 for e in self.false_positives):
            self.false_positives.append(self._row_record(row, self._fp_img_path(row)))
        img.save(self._fp_img_path(row), format='JPEG', quality=92)
        self.remove_pending(row)

    def unmark_false_positive(self, row: pd.Series):
        id1, id2 = int(row['id1']), int(row['id2'])
        self.false_positives = [
            e for e in self.false_positives
            if not (e['id1'] == id1 and e['id2'] == id2)
        ]
        p = self._fp_img_path(row)
        if p.exists(): p.unlink()

    def is_false_positive(self, row: pd.Series) -> bool:
        id1, id2 = int(row['id1']), int(row['id2'])
        return any(e['id1'] == id1 and e['id2'] == id2 for e in self.false_positives)

    # ── Posibles mergers ──────────────────────────────────────────────────────

    def mark_possible_merger(self, row: pd.Series, img: Image.Image):
        id1, id2 = int(row['id1']), int(row['id2'])
        if not any(e['id1'] == id1 and e['id2'] == id2 for e in self.possible_mergers):
            self.possible_mergers.append(self._row_record(row, self._pm_img_path(row)))
        img.save(self._pm_img_path(row), format='JPEG', quality=92)
        self.remove_pending(row)

    def unmark_possible_merger(self, row: pd.Series):
        id1, id2 = int(row['id1']), int(row['id2'])
        self.possible_mergers = [
            e for e in self.possible_mergers
            if not (e['id1'] == id1 and e['id2'] == id2)
        ]
        p = self._pm_img_path(row)
        if p.exists(): p.unlink()

    def is_possible_merger(self, row: pd.Series) -> bool:
        id1, id2 = int(row['id1']), int(row['id2'])
        return any(e['id1'] == id1 and e['id2'] == id2 for e in self.possible_mergers)

    # ── Pares confirmados ─────────────────────────────────────────────────────

    def mark_confirmed_pair(self, row: pd.Series, img: Image.Image):
        id1, id2 = int(row['id1']), int(row['id2'])
        if not any(e['id1'] == id1 and e['id2'] == id2 for e in self.confirmed_pairs):
            self.confirmed_pairs.append(self._row_record(row, self._pair_img_path(row)))
        img.save(self._pair_img_path(row), format='JPEG', quality=92)
        self.remove_pending(row)

    def unmark_confirmed_pair(self, row: pd.Series):
        id1, id2 = int(row['id1']), int(row['id2'])
        self.confirmed_pairs = [
            e for e in self.confirmed_pairs
            if not (e['id1'] == id1 and e['id2'] == id2)
        ]
        p = self._pair_img_path(row)
        if p.exists(): p.unlink()

    def is_confirmed_pair(self, row: pd.Series) -> bool:
        id1, id2 = int(row['id1']), int(row['id2'])
        return any(e['id1'] == id1 and e['id2'] == id2 for e in self.confirmed_pairs)


print('PairValidator definido.')

PairValidator definido.


## 4. Interfaz interactiva

In [5]:
class InspectorUI:
    """
    Grid 4×2 con tres modos:
      - Navegación normal (modo='normal')
      - Reintento de imágenes fallidas (modo='retry')
    Los checkboxes se deshabilitan cuando no hay imagen.
    """

    def __init__(self, validator: PairValidator,
                 page_size: int = PAGE_SIZE,
                 grid_cols: int = GRID_COLS,
                 output_csv: str = OUTPUT_CSV,
                 output_csv_pm: str = OUTPUT_CSV_PM,
                 output_csv_par: str = OUTPUT_CSV_PAR):
        self.v              = validator
        self.page_size      = page_size
        self.grid_cols      = grid_cols
        self.output_csv     = output_csv
        self.output_csv_pm  = output_csv_pm
        self.output_csv_par = output_csv_par
        self._page_imgs     = {}
        self._retry_mode    = False
        self._retry_offset  = 0
        self._build_layout()

    def _build_layout(self):
        self.status_label = widgets.HTML(value=self._status_text())

        self.btn_prev   = widgets.Button(description='◀ Anterior',
                                          button_style='info',
                                          layout=widgets.Layout(width='130px'))
        self.btn_next   = widgets.Button(description='Siguiente ▶',
                                          button_style='primary',
                                          layout=widgets.Layout(width='130px'))
        self.btn_export = widgets.Button(description='📥 Exportar CSV',
                                          button_style='warning',
                                          layout=widgets.Layout(width='150px'))
        self.btn_retry  = widgets.Button(description='🔄 Reintentar fallidas',
                                          button_style='danger',
                                          layout=widgets.Layout(width='180px'))
        self.btn_back_normal = widgets.Button(description='↩ Volver al catálogo',
                                               button_style='',
                                               layout=widgets.Layout(width='180px'))

        self.btn_prev.on_click(self._on_prev)
        self.btn_next.on_click(self._on_next)
        self.btn_export.on_click(self._on_export)
        self.btn_retry.on_click(self._on_enter_retry)
        self.btn_back_normal.on_click(self._on_exit_retry)

        self.nav_bar  = widgets.HBox(
            [self.btn_prev, self.btn_next, self.btn_export, self.btn_retry],
            layout=widgets.Layout(gap='8px', margin='8px 0'))
        self.out_grid   = widgets.Output()
        self.out_status = widgets.Output()
        self.main_box   = widgets.VBox(
            [self.status_label, self.nav_bar, self.out_grid, self.out_status])

        self._prefetch_future   = None
        self._prefetch_rows     = []
        self._prefetch_executor = None

    def _status_text(self) -> str:
        idx   = self.v.current_index
        total = len(self.v.df)
        fp    = len(self.v.false_positives)
        pm    = len(self.v.possible_mergers)
        cp    = len(self.v.confirmed_pairs)
        pr    = len(self.v.pending_retry)
        pct   = 100 * idx / total if total else 0
        mode  = ' &nbsp;<b style="color:#aaf">[MODO REINTENTO]</b>' if self._retry_mode else ''
        return (f'<b>Revisados:</b> {idx:,} / {total:,} ({pct:.1f}%)'
                f'&nbsp;&nbsp;|&nbsp;&nbsp;'
                f'<b style="color:#e55">Falsos positivos:</b> {fp}'
                f'&nbsp;&nbsp;|&nbsp;&nbsp;'
                f'<b style="color:#f90">Posibles mergers:</b> {pm}'
                f'&nbsp;&nbsp;|&nbsp;&nbsp;'
                f'<b style="color:#6f6">Pares confirmados:</b> {cp}'
                f'&nbsp;&nbsp;|&nbsp;&nbsp;'
                f'<b style="color:#aaf">Sin imagen:</b> {pr}'
                + mode)

    def _coord_html(self, ra: float, dec: float) -> widgets.HTML:
        return widgets.HTML(
            value=(
                f'<div style="font-family:monospace;font-size:11px;'
                f'text-align:center;padding:2px 0;">'
                f'{ra:.5f} {dec:.5f}'
                f'</div>'
            ),
            layout=widgets.Layout(width=f'{IMG_SIZE_PX}px', margin='2px 0 0 0')
        )

    def _build_cell(self, raw, row: pd.Series) -> widgets.VBox:
        """Construye la celda de una imagen. Deshabilita checkboxes si no hay imagen."""
        has_img = raw is not None
        img = annotate_image(raw, row, self.v.rp_col) if has_img else make_error_tile()

        buf = BytesIO()
        img.save(buf, format='JPEG', quality=88)
        img_w = widgets.Image(value=buf.getvalue(), format='jpeg',
                               width=IMG_SIZE_PX, height=IMG_SIZE_PX)

        ra_mid  = (row['ra1'] + row['ra2']) / 2.0
        dec_mid = (row['dec1'] + row['dec2']) / 2.0
        coord_w = self._coord_html(ra_mid, dec_mid)

        cb_fp = widgets.Checkbox(
            value=self.v.is_false_positive(row),
            description='Falso positivo',
            disabled=not has_img,
            indent=False,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width=f'{IMG_SIZE_PX}px'))

        cb_pm = widgets.Checkbox(
            value=self.v.is_possible_merger(row),
            description='Posible merger',
            disabled=not has_img,
            indent=False,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width=f'{IMG_SIZE_PX}px'))

        cb_par = widgets.Checkbox(
            value=self.v.is_confirmed_pair(row),
            description='Par confirmado',
            disabled=not has_img,
            indent=False,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width=f'{IMG_SIZE_PX}px'))

        # Registrar como pendiente si no cargó
        if not has_img:
            self.v.add_pending(row)

        def _make_callbacks(r, annotated_img, cb_fp_, cb_pm_, cb_par_):
            def _on_fp(change):
                if change['new']:
                    self.v.mark_false_positive(r, annotated_img)
                else:
                    self.v.unmark_false_positive(r)
                self.status_label.value = self._status_text()

            def _on_pm(change):
                if change['new']:
                    self.v.mark_possible_merger(r, annotated_img)
                else:
                    self.v.unmark_possible_merger(r)
                self.status_label.value = self._status_text()

            def _on_par(change):
                if change['new']:
                    self.v.mark_confirmed_pair(r, annotated_img)
                else:
                    self.v.unmark_confirmed_pair(r)
                self.status_label.value = self._status_text()

            cb_fp_.observe(_on_fp,   names='value')
            cb_pm_.observe(_on_pm,   names='value')
            cb_par_.observe(_on_par, names='value')

        _make_callbacks(row.copy(), img, cb_fp, cb_pm, cb_par)

        return widgets.VBox(
            [img_w, coord_w, cb_fp, cb_pm, cb_par],
            layout=widgets.Layout(margin='4px', align_items='center'))

    def _render_rows(self, row_list, raw_imgs):
        """Renderiza la grilla a partir de listas ya descargadas."""
        cells = []
        for raw, row_dict in zip(raw_imgs, row_list):
            row = pd.Series(row_dict)
            cells.append(self._build_cell(raw, row))

        while len(cells) % self.grid_cols != 0:
            cells.append(widgets.VBox([widgets.Label('—')]))

        rows_w = [widgets.HBox(cells[r*self.grid_cols:(r+1)*self.grid_cols])
                  for r in range(len(cells) // self.grid_cols)]

        with self.out_grid:
            clear_output(wait=True)
            display(widgets.VBox(rows_w))

        self.status_label.value = self._status_text()

    def _render_page(self):
        """Modo normal: navega el catálogo completo. Usa prefetch si está disponible."""
        page = self.v.get_page(self.page_size)
        if page.empty:
            with self.out_grid:
                clear_output(wait=True)
                print('✅ Has revisado todos los pares del catálogo.')
            return
    
        # ── ¿Tenemos prefetch listo? ──────────────────────────────────────────
        prefetch_ready = (
            hasattr(self, '_prefetch_future') and
            self._prefetch_future is not None and
            self._prefetch_future.done()
        )
    
        if prefetch_ready:
            raw_imgs = self._prefetch_future.result()
            row_list = self._prefetch_rows
            self._prefetch_future = None  # limpiar caché
        else:
            # Descarga normal (primera página o prefetch no listo aún)
            with self.out_status:
                clear_output(wait=True)
                print(f'Descargando {len(page)} imágenes…')
    
            row_list = []
            for _, row in page.iterrows():
                sep = float(row.get('sep_arcsec',
                            np.hypot(
                                (row['ra1'] - row['ra2']) *
                                np.cos(np.radians((row['dec1'] + row['dec2']) / 2)) * 3600,
                                (row['dec1'] - row['dec2']) * 3600
                            )))
                row_list.append({**row.to_dict(), 'sep_arcsec': sep})
    
            raw_imgs = fetch_page_parallel(row_list)
    
        # ── Renderizar ────────────────────────────────────────────────────────
        self._render_rows(row_list, raw_imgs)
    
        with self.out_status:
            clear_output(wait=True)
            start = self.v.current_index - self.page_size + 1  # ya avanzó en _on_next
            end   = self.v.current_index
            # Corregir para primera página (current_index no avanzó aún)
            if not prefetch_ready or start < 1:
                start = self.v.current_index + 1
                end   = min(self.v.current_index + self.page_size, len(self.v.df))
            print(f'Pares {start}–{end} de {len(self.v.df):,}')
    
        # ── Lanzar prefetch de la siguiente página ────────────────────────────
        self._launch_prefetch()

    def _render_retry_page(self):
        """Modo retry: muestra solo los pares pendientes de esta página."""
        pending_df = self.v.get_pending_df()
        if pending_df.empty:
            with self.out_grid:
                clear_output(wait=True)
                print('✅ No quedan imágenes pendientes.')
            self._on_exit_retry(None)
            return

        total_p = len(pending_df)
        page = pending_df.iloc[self._retry_offset :
                               min(self._retry_offset + self.page_size, total_p)]

        with self.out_status:
            clear_output(wait=True)
            print(f'Reintentando {len(page)} imágenes ({self._retry_offset+1}–'
                  f'{self._retry_offset+len(page)} de {total_p} pendientes)…')

        row_list = []
        for _, row in page.iterrows():
            sep = float(np.hypot(
                (row['ra1']-row['ra2']) * np.cos(np.radians((row['dec1']+row['dec2'])/2)) * 3600,
                (row['dec1']-row['dec2']) * 3600))
            row_list.append({**row.to_dict(), 'sep_arcsec': sep})

        raw_imgs = fetch_page_parallel(row_list)

        # Los que sí cargaron esta vez se eliminan de pending
        for raw, rd in zip(raw_imgs, row_list):
            if raw is not None:
                self.v.remove_pending(pd.Series(rd))

        self._render_rows(row_list, raw_imgs)

        with self.out_status:
            clear_output(wait=True)
            remaining = len(self.v.pending_retry)
            print(f'Pendientes restantes: {remaining}')

    # ── Navegación ────────────────────────────────────────────────────────────

    def _on_next(self, _):
        if self._retry_mode:
            self._retry_offset = min(
            self._retry_offset + self.page_size,
            max(0, len(self.v.pending_retry) - 1))
            self.v.save_progress()
            self._render_retry_page()
        else:
            self.v.advance(self.page_size)
            self.v.save_progress()
            self._render_page()
            # Pre-fetch de la página siguiente en background
            self._launch_prefetch()

    def _launch_prefetch(self):
        """Descarga la siguiente página en background mientras el usuario clasifica."""
        next_start = self.v.current_index
        next_end   = min(next_start + self.page_size, len(self.v.df))
        if next_start >= len(self.v.df):
            return
        page = self.v.df.iloc[next_start:next_end]
        row_list = []
        for _, row in page.iterrows():
            sep = float(row.get('sep_arcsec',
                        np.hypot((row['ra1']-row['ra2']) *
                                 np.cos(np.radians((row['dec1']+row['dec2'])/2)) * 3600,
                                 (row['dec1']-row['dec2']) * 3600)))
            row_list.append({**row.to_dict(), 'sep_arcsec': sep})
        # Guardar en caché para usarla en el siguiente _render_page
        self._prefetch_executor = ThreadPoolExecutor(max_workers=N_WORKERS)
        self._prefetch_future   = self._prefetch_executor.submit(
            fetch_page_parallel, row_list)
        self._prefetch_rows     = row_list

    def _on_prev(self, _):
        if self._retry_mode:
            self._retry_offset = max(self._retry_offset - self.page_size, 0)
            self._render_retry_page()
        else:
            self.v.go_back(self.page_size)
            self._render_page()

    def _on_enter_retry(self, _):
        if not self.v.pending_retry:
            with self.out_status:
                clear_output(wait=True)
                print('No hay imágenes pendientes.')
            return
        self._retry_mode   = True
        self._retry_offset = 0
        # Cambiar barra de navegación
        self.nav_bar.children = [
            self.btn_prev, self.btn_next,
            self.btn_back_normal, self.btn_export]
        self.status_label.value = self._status_text()
        self._render_retry_page()

    def _on_exit_retry(self, _):
        self._retry_mode = False
        self.nav_bar.children = [
            self.btn_prev, self.btn_next,
            self.btn_export, self.btn_retry]
        self.status_label.value = self._status_text()
        self._render_page()

    def _on_export(self, _):
        self.v.save_progress()
        self.v.export_csv(self.output_csv, self.output_csv_pm, self.output_csv_par)
        with self.out_status:
            clear_output(wait=True)
            for label, d in [('FP', self.v.fp_img_dir),
                              ('Mergers', self.v.pm_img_dir),
                              ('Pares', self.v.pair_img_dir)]:
                n = len(list(Path(d).glob('*.jpg')))
                print(f'{label}: {n} imágenes en {d}/')

    def run(self):
        display(self.main_box)
        self._render_page()


print('InspectorUI definido.')

InspectorUI definido.


## 5. Inicializar y lanzar

In [6]:
validator = PairValidator(
    catalog_path  = CATALOG_PATH,
    progress_file = PROGRESS_FILE,
    fp_img_dir    = FP_IMG_DIR,
    pm_img_dir    = PM_IMG_DIR,
    pair_img_dir  = PAIR_IMG_DIR,
    rp_max_kpc    = RP_MAX_KPC,
)

Filtro rp < 12.0 kpc: 87,731 → 22,928 pares
Revisados: 0  |  FP: 0  |  Mergers: 0  |  Pares confirmados: 0  |  Pendientes retry: 0


In [7]:
ui = InspectorUI(
    validator,
    output_csv     = OUTPUT_CSV,
    output_csv_pm  = OUTPUT_CSV_PM,
    output_csv_par = OUTPUT_CSV_PAR,
)
ui.run()

---
## 6. Exportar catálogo limpio (post-validación)

In [8]:
def remove_false_positives(catalog_path, fp_csv_path, output_path):
    df = pd.read_parquet(catalog_path)
    fp = pd.read_csv(fp_csv_path)
    fp_pairs = set(zip(fp['id1'], fp['id2']))
    mask     = df.apply(lambda r: (int(r['id1']), int(r['id2'])) not in fp_pairs, axis=1)
    df_clean = df[mask].reset_index(drop=True)
    print(f'Original: {len(df):,}  |  FP: {len(fp):,}  |  Limpio: {len(df_clean):,}')
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_parquet(output_path, index=False)
    print(f'Guardado → {output_path}')
    return df_clean

# Descomentar cuando termines la revisión:
# df_clean = remove_false_positives(
#     catalog_path = CATALOG_PATH,
#     fp_csv_path  = OUTPUT_CSV,
#     output_path  = 'outputs/catalogs/pairs_clean.parquet',
# )

---
## 7. Dataset ML — estructura de archivos

Las imágenes guardadas en `outputs/fp_images/` son los **ejemplos positivos** (falsos positivos = deblending erróneo).

Para entrenar un clasificador binario necesitarás también ejemplos negativos (pares reales confirmados). Estructura sugerida:

```
outputs/
  fp_images/          ← clase 0: falsos positivos  (este notebook los genera)
  real_images/        ← clase 1: pares reales confirmados (a generar manualmente)
  catalogs/
    false_positives.csv
    pairs_clean.parquet
```

Cada imagen es un JPEG de 200×200 px con los círculos de identificación anotados.